In [1]:
import logging
import os
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType
from pyspark.sql import functions as f

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder
from pyspark.ml.tuning import CrossValidator
from pyspark.ml import Pipeline

from xgboost.spark import SparkXGBRegressor

Competizione Kaggle: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques
In questo notebook il nostro obiettivo sarà quello di sviluppare un modello che, a partire da un dataset contenente una serie di informazioni su una collezione di abitazioni (Ames Housing dataset), sia capace di predire il prezzo di vendita delle case considerate. Per realizzare questo compito, andremo dapprima a lavorare sui due dataset che vengono forniti dalla competizione, imputando eventuali valori nulli ed effettuando in generale un'operazione di data cleaning. Quindi, addestreremo un regressore (inizialmente utilizzando l'algoritmo RandomForest e successivamente utilizzando l'algoritmo XGBoost) capace di effettuare le previsioni sul prezzo di vendita delle abitazioni. 
Come richiesto dalla competizione Kaggle, utilizzeremo il training set per addestrare il nostro modello, quindi analizzeremo le performance di generalizzazione del nostro regressore sul dataset di test, utilizzando il sito della competizione per ottenere i risultati delle nostre previsioni. In entrambi casi, utilizzeremo la radice quadrata dell'errore quadratico medio tra il logaritmo della nostra previsione e il logaritmo del prezzo reale di vendita della casa come metrica, come richiesto dalla competizione Kaggle. 

In [2]:
# Percorsi di input e output. In origine il progetto leggeva i dati da HDFS
# (es. "hdfs://localhost:9000/spark/progetto/input"); per eseguirlo in locale
# basta lasciare le cartelle "data" e "output".
DATA_DIR = "data"
OUTPUT_DIR = "output"

spark = SparkSession \
    .builder \
    .master("local[4]") \
    .appName("Spark House Prices") \
    .config("spark.sql.ansi.enabled", "false") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()
# ANSI disattivato: il notebook confronta anche colonne numeriche con la stringa "NA"
# (comportamento predefinito di Spark 3.x, con cui è stato sviluppato il progetto)
spark.sparkContext.setLogLevel("ERROR")
logging.getLogger("XGBoost-PySpark").setLevel(logging.WARNING)


Carichiamo i dataset di train e di test in dei dataframe e analizziamo la struttura dei dati con cui andremo a lavorare.

In [3]:
trainDF = spark \
    .read \
    .csv(f"{DATA_DIR}/train.csv", header=True, inferSchema=True)

testDF = spark \
    .read \
    .csv(f"{DATA_DIR}/test.csv", header=True, inferSchema=True)


In [4]:
print("Training set")
print(f"numero di righe: {trainDF.count()}")
print(f"numero di colonne: {len(trainDF.columns)}")
display(trainDF)
trainDF.printSchema()

print("\n\nTest set")
print(f"numero di righe: {testDF.count()}")
print(f"numero di colonne: {len(testDF.columns)}")
display(testDF)
testDF.printSchema()

Training set


numero di righe: 1460
numero di colonne: 81


DataFrame[Id: int, MSSubClass: int, MSZoning: string, LotFrontage: string, LotArea: int, Street: string, Alley: string, LotShape: string, LandContour: string, Utilities: string, LotConfig: string, LandSlope: string, Neighborhood: string, Condition1: string, Condition2: string, BldgType: string, HouseStyle: string, OverallQual: int, OverallCond: int, YearBuilt: int, YearRemodAdd: int, RoofStyle: string, RoofMatl: string, Exterior1st: string, Exterior2nd: string, MasVnrType: string, MasVnrArea: string, ExterQual: string, ExterCond: string, Foundation: string, BsmtQual: string, BsmtCond: string, BsmtExposure: string, BsmtFinType1: string, BsmtFinSF1: int, BsmtFinType2: string, BsmtFinSF2: int, BsmtUnfSF: int, TotalBsmtSF: int, Heating: string, HeatingQC: string, CentralAir: string, Electrical: string, 1stFlrSF: int, 2ndFlrSF: int, LowQualFinSF: int, GrLivArea: int, BsmtFullBath: int, BsmtHalfBath: int, FullBath: int, HalfBath: int, BedroomAbvGr: int, KitchenAbvGr: int, KitchenQual: string

root
 |-- Id: integer (nullable = true)
 |-- MSSubClass: integer (nullable = true)
 |-- MSZoning: string (nullable = true)
 |-- LotFrontage: string (nullable = true)
 |-- LotArea: integer (nullable = true)
 |-- Street: string (nullable = true)
 |-- Alley: string (nullable = true)
 |-- LotShape: string (nullable = true)
 |-- LandContour: string (nullable = true)
 |-- Utilities: string (nullable = true)
 |-- LotConfig: string (nullable = true)
 |-- LandSlope: string (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Condition1: string (nullable = true)
 |-- Condition2: string (nullable = true)
 |-- BldgType: string (nullable = true)
 |-- HouseStyle: string (nullable = true)
 |-- OverallQual: integer (nullable = true)
 |-- OverallCond: integer (nullable = true)
 |-- YearBuilt: integer (nullable = true)
 |-- YearRemodAdd: integer (nullable = true)
 |-- RoofStyle: string (nullable = true)
 |-- RoofMatl: string (nullable = true)
 |-- Exterior1st: string (nullable = true)
 |--

numero di righe: 1459
numero di colonne: 80


DataFrame[Id: int, MSSubClass: int, MSZoning: string, LotFrontage: string, LotArea: int, Street: string, Alley: string, LotShape: string, LandContour: string, Utilities: string, LotConfig: string, LandSlope: string, Neighborhood: string, Condition1: string, Condition2: string, BldgType: string, HouseStyle: string, OverallQual: int, OverallCond: int, YearBuilt: int, YearRemodAdd: int, RoofStyle: string, RoofMatl: string, Exterior1st: string, Exterior2nd: string, MasVnrType: string, MasVnrArea: string, ExterQual: string, ExterCond: string, Foundation: string, BsmtQual: string, BsmtCond: string, BsmtExposure: string, BsmtFinType1: string, BsmtFinSF1: string, BsmtFinType2: string, BsmtFinSF2: string, BsmtUnfSF: string, TotalBsmtSF: string, Heating: string, HeatingQC: string, CentralAir: string, Electrical: string, 1stFlrSF: int, 2ndFlrSF: int, LowQualFinSF: int, GrLivArea: int, BsmtFullBath: string, BsmtHalfBath: string, FullBath: int, HalfBath: int, BedroomAbvGr: int, KitchenAbvGr: int, K

root
 |-- Id: integer (nullable = true)
 |-- MSSubClass: integer (nullable = true)
 |-- MSZoning: string (nullable = true)
 |-- LotFrontage: string (nullable = true)
 |-- LotArea: integer (nullable = true)
 |-- Street: string (nullable = true)
 |-- Alley: string (nullable = true)
 |-- LotShape: string (nullable = true)
 |-- LandContour: string (nullable = true)
 |-- Utilities: string (nullable = true)
 |-- LotConfig: string (nullable = true)
 |-- LandSlope: string (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Condition1: string (nullable = true)
 |-- Condition2: string (nullable = true)
 |-- BldgType: string (nullable = true)
 |-- HouseStyle: string (nullable = true)
 |-- OverallQual: integer (nullable = true)
 |-- OverallCond: integer (nullable = true)
 |-- YearBuilt: integer (nullable = true)
 |-- YearRemodAdd: integer (nullable = true)
 |-- RoofStyle: string (nullable = true)
 |-- RoofMatl: string (nullable = true)
 |-- Exterior1st: string (nullable = true)
 |--

Notiamo come i due dataset condividono la stessa struttura, con l'eccezione dell'ultima colonna che indica il prezzo di vendita dell'abitazione (non presente nel dataset di test). Una parte delle colonne è costituita da dati di tipo intero, e una parte da dati stringa che costituiranno attributi categorici.
Andiamo a verificare l'eventuale presenza di valori nulli o mancanti.

In [5]:
def null_values_count(df):
    null_columns_counts=[]
    for i in df.columns:
        null_rows=df.where(col(i).isNull()).count()
        if(null_rows>0):
            tupla=i, null_rows
            null_columns_counts.append(tupla)
    return null_columns_counts

train_null=null_values_count(trainDF)
test_null=null_values_count(testDF)

if not train_null:
    print("Non ci sono valori mancanti nel dataset train")
else:
    print(train_null)
if not test_null:
    print("Non ci sono valori mancanti nel dataset test")
else:
    print(test_null)

Non ci sono valori mancanti nel dataset train
Non ci sono valori mancanti nel dataset test


A prima vista sembrerebbe che nei due dataset non ci siano valori mancanti. In realtà andando a visualizzare i valori notiamo come alcuni valori sono stati indicati come NA per indicare dati mancanti nel caso di variabili numeriche, ma così non è per alcune variabili non numeriche dove invece il valore NA indica uno dei possibili valori della variabile. Dal file data_description.txt infatti apprendiamo come per le variabili: 
"Alley, BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinType2, FireplaceQu, GarageType, GarageFinish, GarageQual, GarageCond, PoolQC, Fence, MiscFeature"
i valori indicati come NA nei due dataset non indicano dati mancanti, ma indicano la non presenza della caratteristica specificata per quella particolare abitazione (ad esempio laddove troviamo il valore NA per la caratteristica PoolQC, questo indica che nell'abitazione specificata non vi è una piscina e non che il dato è mancante). Se avessimo lavorato con un dataframe pandas questo avrebbe importato i valori come NaN, e avremmo quindi dovuto esplicitamente fare in modo che non ci fosse una conversione automatica dei valori nulli. PySpark invece non compie automaticamente quest'azione, preservando i valori NA dove corretti, ma andando a inserire dei valori testuali "NA" all'interno di alcune colonne che dovrebbero essere numeriche e che in questo modo sono invece state riconosciute come stringhe (come nel caso di Lot_Frontage). 
Dobbiamo quindi andare a sostituire i valori NA in questi casi come nulli, e fare un cast per indicare quelle colonne come numeriche.

In [6]:
def na_values_count(df):
    na_columns_counts=[]
    for i in df.columns:
        na_rows=df.where(col(i) == "NA" ).count()
        if(na_rows>0):
            tupla=i, na_rows
            na_columns_counts.append(tupla)
    return na_columns_counts

train_na=na_values_count(trainDF)
test_na=na_values_count(testDF)

if train_na:
    train_na_df = pd.DataFrame(train_na, columns=["Colonna", "Counter NA"])
    print("Valori NA nel dataset train:")
    print(train_na_df)
else:
    print("Non ci sono valori NA nel dataset train")

if test_na:
    test_na_df = pd.DataFrame(test_na, columns=["Colonna", "Counter NA"])
    print("Valori NA nel dataset test:")
    print(test_na_df)
else:
    print("Non ci sono valori NA nel dataset test")

Valori NA nel dataset train:
         Colonna  Counter NA
0    LotFrontage         259
1          Alley        1369
2     MasVnrType           8
3     MasVnrArea           8
4       BsmtQual          37
5       BsmtCond          37
6   BsmtExposure          38
7   BsmtFinType1          37
8   BsmtFinType2          38
9     Electrical           1
10   FireplaceQu         690
11    GarageType          81
12   GarageYrBlt          81
13  GarageFinish          81
14    GarageQual          81
15    GarageCond          81
16        PoolQC        1453
17         Fence        1179
18   MiscFeature        1406
Valori NA nel dataset test:
         Colonna  Counter NA
0       MSZoning           4
1    LotFrontage         227
2          Alley        1352
3      Utilities           2
4    Exterior1st           1
5    Exterior2nd           1
6     MasVnrType          16
7     MasVnrArea          15
8       BsmtQual          44
9       BsmtCond          45
10  BsmtExposure          44
11  BsmtFinType

Di queste quindi, ignorando le categorie per le quali NA è un valore ammissibile, abbiamo dei valori mancanti in:
Per il train set
-   LotFrontage		259             (inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)
-   GarageYrBlt		81              (inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)
-   MasVnrType		8
-   MasVnrArea		8               (inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)
-   Electrical		1
Andiamo ad imputare i valori mancanti e a correggere dove necessario il tipo della varabile considerata laddove ci siano stati errori di importazione.
- per LotFrontage eseguiremo la media per quartiere
- per GarageYrBlt inizialmente si era pensato di impostare un valore pari a 0 per indicare che il valore è assente, dato che è stato utilizzato NA in tutti i casi in cui i valori di GarageType, GarageFinish, GarageQual e GarageCond indicavano la non presenza di un garage. Tuttavia così facendo otterremmo un valore poco significativo rispetto al significato della variabile; infatti avremmo il valore 0 in un contesto in cui è richiesto l'anno di costruzione del garage. 
Dato che in base ad analisi successive si è osservato che non si tratta di una variabile particolarmente significativa per il problema che ci siamo posti, abbiamo quindi optato per non considerare questa variabile.
- per MasVnrType e MasVnrArea il valore NA non indica un valore nullo (il valore nullo è invece indicato con None), dato che non abbiamo particolari informazioni sul come potremmo imputare questo dato possiamo scegliere o di rimuovere le poche righe in questione, oppure semplicemente potremmo selezionare il valore più diffuso nel caso di MasVnrType e conseguentemente 0 nel caso di MasVnrArea.
- per Electrical imposteremo il valore più diffuso per questa categoria nel dataset, ovvero il sistema elettrico rappresentato da SBrkr.

In [7]:
#Andiamo a sostituire i valori NA come valori nulli in LotFrontage e trasformiamo i dati che erano stati erroneamente riconosciuti come stringhe in tipi Interi
trainDF = trainDF.withColumn("LotFrontage", when(col("LotFrontage") == "NA", None).otherwise(col("LotFrontage")))
trainDF = trainDF.withColumn("LotFrontage", col("LotFrontage").cast(IntegerType()))

#trainDF = trainDF.withColumn("GarageYrBlt", when(col("GarageYrBlt") == "NA", None).otherwise(col("GarageYrBlt")))
#trainDF = trainDF.withColumn("GarageYrBlt", col("GarageYrBlt").cast(IntegerType()))

trainDF = trainDF.withColumn("MasVnrArea", when(col("MasVnrArea") == "NA", None).otherwise(col("MasVnrArea")))
trainDF = trainDF.withColumn("MasVnrArea", col("MasVnrArea").cast(IntegerType()))

trainDF = trainDF.withColumn("MasVnrType", when(col("MasVnrType") == "NA", None).otherwise(col("MasVnrType")))
trainDF = trainDF.withColumn("Electrical", when(col("Electrical") == "NA", None).otherwise(col("Electrical")))



In [8]:
#Impostiamo il valore di LotFrontage in base al valore della media del quartiere dell'abitazione considerata
#creiamo un nuovo dataframe contenente la media per quartiere della variabile LotFrontage
neighborhood_avg = trainDF.groupBy("Neighborhood").agg(avg("LotFrontage").alias("LotFrontage_avg"))
#facciamo il join tra i due dataframe
trainDF = trainDF.join(neighborhood_avg, on="Neighborhood", how="left")
#impostiamo i nuovi valori laddove LotFrontage ha un valore nullo
trainDF = trainDF.withColumn("LotFrontage", 
                             when(col("LotFrontage").isNull(), col("LotFrontage_avg")).otherwise(col("LotFrontage")))
#eliminiamo dal dataframe la colonna delle medie per quartiere della variabile LotFrontage
trainDF = trainDF.drop("LotFrontage_avg")
# NB: neighborhood_avg viene riutilizzato anche per il test set, così il test viene imputato
# con statistiche calcolate solo sui dati di training

In [9]:
# Notiamo come il valore mancante di GarageYrBlt corrisponde in tutti i casi ad un valore di garage assente delle altre variabili, per cui impostiamo a 0 il valore di questa variabile
missing_garage = trainDF.filter(
    (col("GarageYrBlt").isNull()) & 
    (col("GarageType") == "NA") &
    (col("GarageFinish") == "NA") &
    (col("GarageQual") == "NA") &
    (col("GarageCond") == "NA")
)

# Visualizza le righe filtrate
#missing_garage.select("GarageYrBlt", "GarageType", "GarageFinish", "GarageQual", "GarageCond").count()

#trainDF = trainDF.withColumn("GarageYrBlt", when(col("GarageYrBlt").isNull(), 0).otherwise(col("GarageYrBlt")))

trainDF=trainDF.drop("GarageYrBlt")


In [10]:
trainDF.groupBy("MasVnrType").count().show()

trainDF= trainDF.withColumn("MasVnrType", when(col("MasVnrType").isNull(), "None").otherwise(col("MasVnrType")))
trainDF= trainDF.withColumn("MasVnrArea", when(col("MasVnrArea").isNull(), 0).otherwise(col("MasVnrArea")))


+----------+-----+
|MasVnrType|count|
+----------+-----+
|      None|  864|
|    BrkCmn|   15|
|      NULL|    8|
|     Stone|  128|
|   BrkFace|  445|
+----------+-----+


In [11]:
#Impostiamo il valore mancante di Electrical in base al valore più diffuso per questa variabile.
trainDF.groupBy("Electrical").count().show()

trainDF= trainDF.withColumn("Electrical", when(col("Electrical").isNull(), "SBrkr").otherwise(col("Electrical")))


+----------+-----+
|Electrical|count|
+----------+-----+
|       Mix|    1|
|     FuseF|   27|
|      NULL|    1|
|     SBrkr| 1334|
|     FuseP|    3|
|     FuseA|   94|
+----------+-----+


In [12]:
# dtypes ci restituite una tupla (colonna, tipo)
dtypes=trainDF.dtypes
quantitative=[]
qualitative=[]
# iteriamo su ogni colonna per dividere le colonne di tipo intero da quelle di tipo stringa
for colonna, tipo in dtypes:
    if tipo != "string" and colonna != "Id" and colonna !="SalePrice":
        quantitative.append(colonna)
    if tipo == "string" and colonna != "GarageYrBlt":
        qualitative.append(colonna)
# eliminiamo dal conto delle variabili quantitative la colonna relativa ad id e quella relativa al prezzo di vendita
print(f"Numero variabili Quantitative: {len(quantitative)}")
print(quantitative)
print(f"\nNumero variabili Qualitative: {len(qualitative)}")
print(qualitative)

Numero variabili Quantitative: 35
['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']
Numero variabili Qualitative: 43
['Neighborhood', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'G

Facciamo lo stesso lavoro di imputazione dei valori nulli e correzione dei tipi delle colonne anche per il test set.
Nel test set abbiamo:
- LotFrontage		227			(inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)
- MSZoning		    4
- Utilities		    2
- Exterior1st		1
- Exterior2nd		1
- MasVnrType		16
- MasVnrArea		15			(inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)
- BsmtFinSF1		1			(inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)
- BsmtFinSF2		1			(inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)
- BsmtUnfSF		    1			(inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)
- TotalBsmtSF		1			(inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)
- BsmtFullBath	    2			(inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)
- BasmtHalfBath	    2			(inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)
- KitchenQual		1						
- Functional		2						
- GarageYrBlt		78			(inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)					
- GarageCars		1			(inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)
- GarageArea		1			(inoltre dobbiamo modificare il tipo della colonna da stringa ad intero)
- SaleType		    1

In [13]:
#Andiamo a sostituire i valori NA come valori nulli in LotFrontage e trasformiamo i dati che erano stati erroneamente riconosciuti come stringhe in tipi Interi
testDF = testDF.withColumn("LotFrontage", when(col("LotFrontage") == "NA", None).otherwise(col("LotFrontage")))
testDF = testDF.withColumn("LotFrontage", col("LotFrontage").cast(IntegerType()))

#Impostiamo il valore di LotFrontage in base alla media del quartiere dell'abitazione considerata.
#Usiamo le medie calcolate sul training set (neighborhood_avg), per non usare statistiche del test set.
#facciamo il join tra i due dataframe
testDF = testDF.join(neighborhood_avg, on="Neighborhood", how="left")
#impostiamo i nuovi valori laddove LotFrontage ha un valore nullo
testDF = testDF.withColumn("LotFrontage", 
                             when(col("LotFrontage").isNull(), col("LotFrontage_avg")).otherwise(col("LotFrontage")))
#eliminiamo dal dataframe la colonna delle medie per quartiere della variabile LotFrontage
testDF = testDF.drop("LotFrontage_avg")

In [14]:
# Dato che non abbiamo particolari indicazioni su come imputare i valori mancanti della classificazione della zona dell'abitazione, e che soluzioni come lo sfruttare il valore più diffuso per il quartiere di appartenenza non sembrano fornire dati particolarmente utili, lasciamo la categoria NA per indicare dei valori sconosciuti. 
testDF.filter(col("MSZoning") == "NA").select("Id", "MSZoning", "Neighborhood").show()

#Per imputare il valore mancante di KitchenQual utilizziamo il valore più diffuso per la categoria
testDF.groupBy("KitchenQual").count().show()
testDF = testDF.withColumn("KitchenQual", when(col("KitchenQual") == "NA", "TA").otherwise(col("KitchenQual")))

#Per imputare i valori mancanti di Functional utilizziamo il valore più diffuso per la categoria
testDF.groupBy("Functional").count().show()
testDF = testDF.withColumn("Functional", when(col("Functional") == "NA", "Typ").otherwise(col("Functional")))

+----+--------+------------+
|  Id|MSZoning|Neighborhood|
+----+--------+------------+
|1916|      NA|      IDOTRR|
|2217|      NA|      IDOTRR|
|2251|      NA|      IDOTRR|
|2905|      NA|     Mitchel|
+----+--------+------------+


+-----------+-----+
|KitchenQual|count|
+-----------+-----+
|         Gd|  565|
|         NA|    1|
|         Ex|  105|
|         Fa|   31|
|         TA|  757|
+-----------+-----+


+----------+-----+
|Functional|count|
+----------+-----+
|      Maj1|    5|
|        NA|    2|
|       Sev|    1|
|      Min2|   36|
|      Min1|   34|
|       Typ| 1357|
|      Maj2|    4|
|       Mod|   20|
+----------+-----+


In [15]:
# Per imputare il valore mancante di Utilities utilizziamo il valore più diffuso per quella variabile ovvero AllPub. 
testDF.groupBy("Utilities").count().show()
testDF = testDF.withColumn("Utilities", when(col("Utilities") == "NA", "AllPub").otherwise(col("Utilities")))

# Per imputare il valore Exterior1st ed Exterior2nd utilizziamo il valore più diffuso nelle due categorie
testDF.filter((col("Exterior1st") == "NA") & (col("Exterior2nd") == "NA")) \
    .select("Id", "Exterior1st", "Exterior2nd").show()
testDF.groupBy("Exterior1st").count().orderBy("count").show()
testDF.groupBy("Exterior2nd").count().orderBy("count").show()
testDF = testDF.withColumn("Exterior1st", when(col("Exterior1st") == "NA", "VinylSd").otherwise(col("Exterior1st")))
testDF = testDF.withColumn("Exterior2nd", when(col("Exterior2nd") == "NA", "VinylSd").otherwise(col("Exterior2nd")))


+---------+-----+
|Utilities|count|
+---------+-----+
|       NA|    2|
|   AllPub| 1457|
+---------+-----+
+----+-----------+-----------+
|  Id|Exterior1st|Exterior2nd|
+----+-----------+-----------+
|2152|         NA|         NA|
+----+-----------+-----------+


+-----------+-----+
|Exterior1st|count|
+-----------+-----+
|         NA|    1|
|    AsphShn|    1|
|     CBlock|    1|
|    BrkComm|    4|
|     Stucco|   18|
|    AsbShng|   24|
|    WdShing|   30|
|    BrkFace|   37|
|    CemntBd|   65|
|    Plywood|  113|
|    Wd Sdng|  205|
|    HdBoard|  220|
|    MetalSd|  230|
|    VinylSd|  510|
+-----------+-----+


+-----------+-----+
|Exterior2nd|count|
+-----------+-----+
|         NA|    1|
|    AsphShn|    1|
|      Stone|    1|
|     CBlock|    2|
|    ImStucc|    5|
|    Brk Cmn|   15|
|    AsbShng|   18|
|     Stucco|   21|
|    BrkFace|   22|
|    Wd Shng|   43|
|    CmentBd|   66|
|    Plywood|  128|
|    Wd Sdng|  194|
|    HdBoard|  199|
|    MetalSd|  233|
|    VinylSd|  510|
+-----------+-----+


In [16]:
# Per imputare il valore di MasVnrType e MasVnrArea, utilizziamo il valore più diffuso nel caso di MasVnrType e di conseguenza 0 nel caso di MasVnrArea (non usiamo la media dato che mettere None nel caso di MasVnrType e un valore diverso da 0 per MasVnrArea non avrebbe senso)
testDF.groupBy("MasVnrType").count().show()

testDF= testDF.withColumn("MasVnrType", when(col("MasVnrType").isNull(), "None").otherwise(col("MasVnrType")))
testDF= testDF.withColumn("MasVnrArea", when(col("MasVnrArea").isNull(), 0).otherwise(col("MasVnrArea")))
testDF = testDF.withColumn("MasVnrArea", col("MasVnrArea").cast(IntegerType()))


+----------+-----+
|MasVnrType|count|
+----------+-----+
|      None|  878|
|    BrkCmn|   10|
|        NA|   16|
|     Stone|  121|
|   BrkFace|  434|
+----------+-----+


In [17]:
# Notiamo come il valore mancante di BsmtFinSF1, BsmtFinSF2, BsmtUnfSF, TotalBsmtSF,BsmtFullBath, BsmtHalfBath corrisponde in tutti i casi ad un valore di basement assente delle altre variabili, per cui impostiamo a 0 il valore di questa variabile
missing_bsmt = testDF.filter(
    (col("BsmtFullBath") == "NA") &
    (col("BsmtHalfBath") == "NA")
)

# Visualizza le righe filtrate
missing_bsmt.select("BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath", "BsmtFinType1").show()

# Impostiamo a 0 i valori NA in ciascuna colonna
columns_to_update = ["BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath"]
for column in columns_to_update:
    testDF = testDF.withColumn(column, when(col(column) == "NA", 0).otherwise(col(column)))
    testDF = testDF.withColumn(column, col(column).cast(IntegerType()))



+----------+----------+---------+-----------+------------+------------+------------+
|BsmtFinSF1|BsmtFinSF2|BsmtUnfSF|TotalBsmtSF|BsmtFullBath|BsmtHalfBath|BsmtFinType1|
+----------+----------+---------+-----------+------------+------------+------------+
|        NA|        NA|       NA|         NA|          NA|          NA|          NA|
|         0|         0|        0|          0|          NA|          NA|          NA|
+----------+----------+---------+-----------+------------+------------+------------+


In [18]:
# Per GarageYrBlt notiamo che in 76 casi su 78 il dato mancante coincide con il valore NA per la variabile GarageType (che indica assenza di garage). Tuttavia dato che impostare un valore pari a 0 non avrebbe senso per il significato della variabile che dovrebbe indicare un anno per la costruzione del garage, scegliamo come fatto nel dataset di training di rimuovere completamente la colonna. 
missing_garage = testDF.filter(
    (col("GarageYrBlt")=="NA") & 
    (col("GarageType") == "NA")
).count()
testDF=testDF.drop("GarageYrBlt")

# Notiamo la stessa cosa anche per GarageCars e GarageArea, per cui sostituiamo i valori NA con 0 e modifichiamo il tipo della colonna al tipo intero
testDF = testDF.withColumn("GarageCars", when(col("GarageCars") == "NA", 0).otherwise(col("GarageCars")))
testDF = testDF.withColumn("GarageCars", col("GarageCars").cast(IntegerType()))

testDF = testDF.withColumn("GarageArea", when(col("GarageArea") == "NA", 0).otherwise(col("GarageArea")))
testDF = testDF.withColumn("GarageArea", col("GarageArea").cast(IntegerType()))

In [19]:
#Per imputare il valore mancante di SaleType utilizziamo il valore più diffuso per la categoria
testDF.groupBy("SaleType").count().show()
testDF = testDF.withColumn("SaleType", when(col("SaleType") == "NA", "WD").otherwise(col("SaleType")))


+--------+-----+
|SaleType|count|
+--------+-----+
|   ConLD|   17|
|     COD|   44|
|     Con|    3|
|      NA|    1|
|     Oth|    4|
|   ConLI|    4|
|     CWD|    8|
|   ConLw|    3|
|      WD| 1258|
|     New|  117|
+--------+-----+


Uniamo trainDF e testDF in un unico dataframe in maniera tale da poter utilizzare lo StringIndexer per poter lavorare anche con i dati non numerici con gli algoritmi che ci serviranno. è importante lavorare su un unico dataset in maniera tale da utilizzare la stessa mappatura per le caratteristiche sia nel train set che nel test set.

In [20]:
# Creiamo un unico dataframe che contenga tutte le colonne che ci interessano
labelDF = trainDF.select(qualitative).unionByName(testDF.select(qualitative))

# Per ogni colonna qualitativa creiamo un oggetto StringIndexer per indicizzare le categorie qualitative in categorie con indici numerici. 
indexers = [StringIndexer(inputCol=column, outputCol=column+"_index").fit(labelDF) for column in qualitative]

# Aggiungiamo le nuove categorie indicizzate nei due dataframe
for indexer in indexers:
    trainDF=indexer.transform(trainDF)
    testDF=indexer.transform(testDF)

In [21]:
print(trainDF.columns)

['Neighborhood', 'Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', 'PavedDrive', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'PoolQC', 'Fence', 'Mis

Avendo inidicizzato le categorie qualitative, dobbiamo ora modificare il formato delle feature per renderlo adatto agli algoritmi che andremo ad utilizzare. Dato che molti algoritmi di Machine Learning richiedono che tutte le feature siano organizzate in un unico vettore, combiniamo le feature in un'unica colonna con VectorAssembler.

In [22]:
feature_columns=quantitative+[column+"_index" for column in qualitative]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

A questo punto possiamo provare ad addestrare una versione preliminare del nostro regressore. Per iniziare addestreremo l'algoritmo considerando tutte le features del dataset. Non ci aspettiamo che questo porti a risultati particolarmente soddisfacenti; lavorando con così tante features è facile introdurre un rumore significativo nell'addestramento, inoltre ridurre il numero delle features ci aiuta ad evitare problemi di overfitting, migliorando le capacità di generalizzazione ed evitando che l'algoritmo si adatti eccessivamente ai dati di training, oltre a migliorare l'efficienza dell'addestramento. Procedere in questo modo però ci servirà per poter trarre delle prime considerazioni su quali siano le feature più importanti da considerare.

L'algoritmo che utilizzeremo inizialmente è il RandomForestRegressor, imposteremo un seed per fare in modo di poter confrontare risultati di volta in volta. Si è scelto di utilizzare il RandomForest perchè si tratta di un algoritmo che si adatta bene a dataset con molte features ed è in grado di gestire sia feature numeriche che qualitative senza bisogno di effettuare un particolare lavoro di normalizzazione dei dati. Inoltre, questo tipo di algoritmo è relativamente insensibile al rumore nei dati, dato che ogni albero utilizza un sottoinsieme casuale delle features e dei dati; è abbastanza semplice da ottimizzare avendo un numero non elevato di iperparametri da gestire e, soprattutto, ci permette di ottenere una misura dell'importanza delle features che ci servirà per selezionare le variabili più importanti da considerare per l'addestramento.
Avendo stabilito l'estimator che utilizzeremo, rivolgiamo la nostra attenzione all'evaluator che si occuperà di misurare i risultati del modello. Il testo del problema Kaggle ci consiglia di utilizzare la radice quadrata dell'errore quadratico medio calcolato sul logaritmo del prezzo previsto e sul logaritmo del prezzo reale come metrica di valutazione in base alla quale fare in modo che l'algoritmo ottimizzi i risultati. Impostiamo quindi questa metrica nel RegressionEvaluator.
Avendo eseguito diversi test utilizzando come metrica il semplice RMSE tra prezzo previsto e prezzo reale, si è visto come in effetti i risultati sono leggermente migliori nel primo caso, per cui continuiamo la nostra analisi utilizzando i logaritmi come metrica, ma riportando e calcolando anche i valori di RMSE semplici per maggiore chiarezza in maniera tale da poter valutare meglio i risultati ottenuti.

In [23]:
# Aggiungiamo la colonna logaritmica log_SalePrice che ci servirà per calcolare l'errore
trainDF=trainDF.withColumn("log_SalePrice", f.log1p(trainDF["SalePrice"]))

# Dividiamo il dataset in una parte destinata al training ed una parte per il test
train_data, test_data = trainDF.randomSplit([0.8, 0.2], seed=48)

# Utilizziamo il regressore Random Forest come Estimator, impostando il vettore delle feature e la label che vogliamo predire.
randomForest = RandomForestRegressor(featuresCol="features", labelCol="log_SalePrice", numTrees=100, maxBins=128, seed=48)

# Impostiamo la pipeline di trasformazione specificando gli stages di lavoro che dovremo seguire.
# assembliamo le features in un unico vettore con l'assembler, ed utilizziamo l'estimator randomForest per addestrare un modello utilizzando il vettore delle features per prevedere SalePrice. 
pipeline = Pipeline(stages=[assembler, randomForest])

# Utilizzando il metodo fit della pipeline, otterremo un Transformer, che utilizzeremo per predire i prezzi delle case.
model = pipeline.fit(train_data)

# Utilizzando il metodo transform del Transformer prevediamo i prezzi delle case della parte di dataset che abbiamo riservato al testing.
predictions = model.transform(test_data)

# Dobbiamo quindi calcolare il logaritmo delle previsioni che abbiamo effettuato per poter valutare il modello utilizzando la radice dell'errore quadratico medio calcolato sul logaritmo del prezzo previsto e sul logaritmo del prezzo effettivo. 
predictions = predictions.withColumn("exp_prediction", f.expm1(predictions["prediction"]))

# Utilizziamo il RegressionEvaluator per calcolare la radice dell'errore quadratico medio sui logaritmi delle previsioni. Questo ci consente di trattare in un modo uniforme sia gli errori sulle case più economiche che su quelle più costose rispetto alla nostra predizione.  
evaluator = RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) between Logarithms of SalePrice: {rmse}")

evaluator_raw = RegressionEvaluator(labelCol="SalePrice", predictionCol="exp_prediction", metricName="rmse")
rmse_raw = evaluator_raw.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on original SalePrice: {rmse_raw}")

Root Mean Squared Error (RMSE) between Logarithms of SalePrice: 0.15057605289258966


Root Mean Squared Error (RMSE) on original SalePrice: 32408.165207262486


Il primo algoritmo che abbiamo addestrato per ottenere l'importanza delle features ci ha dato un risultato non eccessivamente soddisfacente come previsto. Va anche considerato che non abbiamo ancora fatto il tuning degli iperparametri e ci siamo concentrati sull'ottenere un'esecuzione che fosse la più rapida possibile. Estraiamo ora l'importanza delle features. 

In [24]:
# Estraiamo adesso l'importanza delle feature da questo primo addestramento dell'algoritmo
importances = model.stages[-1].featureImportances  

# e mappiamo le importanze con le colonne corrispondenti nel dataset
feature_importances = list(zip(feature_columns, importances.toArray()))

# ordiniamo le features per importanza e selezioniamo quelle con un valore maggiore di 0.01
sorted_feature_importances = sorted(feature_importances, key=lambda x: x[1], reverse=True)

important_feature=[]
for feature, importance in sorted_feature_importances:
    print(f"Feature: {feature}, Importanza: {importance}")
    if(importance > 0.01):
        important_feature.append(feature)
print(important_feature)

Feature: OverallQual, Importanza: 0.24744375165191815
Feature: Neighborhood_index, Importanza: 0.17418232015698035
Feature: GrLivArea, Importanza: 0.13200454989304325
Feature: ExterQual_index, Importanza: 0.08152767982253892
Feature: YearBuilt, Importanza: 0.05336322260987689
Feature: KitchenQual_index, Importanza: 0.04075486812240195
Feature: TotalBsmtSF, Importanza: 0.03605160201797318
Feature: GarageCars, Importanza: 0.030578483755751656
Feature: 1stFlrSF, Importanza: 0.02548315902909578
Feature: GarageArea, Importanza: 0.022368079248857572
Feature: BsmtQual_index, Importanza: 0.01698740053189819
Feature: CentralAir_index, Importanza: 0.013944855363687378
Feature: BsmtFinSF1, Importanza: 0.011867137555981762
Feature: LotArea, Importanza: 0.009261622080586151
Feature: FireplaceQu_index, Importanza: 0.008950175991805727
Feature: GarageType_index, Importanza: 0.008608412264327151
Feature: 2ndFlrSF, Importanza: 0.007268624609108326
Feature: MSZoning_index, Importanza: 0.0072284438137536

Con questa prima esecuzione siamo riusciti a ridurre il numero delle features da considerare a 13. Come già accennato, ridurre il numero delle features è molto importante per svariati motivi, tra i quali sicuramente spicca la migliore capacità di generalizzazione che possiamo ottenere in questo modo. Infatti, se abbiamo troppe features rispetto al numero totale di campioni, il nostro modello rischia di apprendere del rumore o di concentrarsi su dettagli irrilevanti, piuttosto che riuscire ad astrarre le relazioni generali tra i dati.

Proviamo quindi ad effettuare un nuovo addestramento su questo sottoinsieme di features, utilizzando questa volta paramGrid per cercare di ottimizzare i migliori iperparametri da utilizzare nel nostro algoritmo. 

In [25]:
# Organizziamo tutte le features in un unico vettore come richiesto dall'algoritmo Random Forest
assembler = VectorAssembler(inputCols=important_feature, outputCol="features")

# Aggiungiamo la colonna logaritmica log_SalePrice che ci servirà per calcolare l'errore
trainDF=trainDF.withColumn("log_SalePrice", f.log1p(trainDF["SalePrice"]))

# Dividiamo il dataset in una parte destinata al training ed una parte per il test
train_data, test_data = trainDF.randomSplit([0.8, 0.2], seed=48)

# Utilizziamo il regressore Random Forest come Estimator, impostando il vettore delle feature e la label che vogliamo predire.
randomForest = RandomForestRegressor(featuresCol="features", labelCol="log_SalePrice", seed=48)

# Impostiamo la pipeline di trasformazione specificando gli stages di lavoro che dovremo seguire.
# assembliamo le features in un unico vettore con l'assembler, ed utilizziamo l'estimator randomForest per addestrare un modello utilizzando il vettore delle features per prevedere SalePrice. 
pipeline = Pipeline(stages=[assembler, randomForest])

# Grid dei parametri per il tuning
paramGrid = ParamGridBuilder()\
    .addGrid(randomForest.numTrees, [50, 100, 200])\
    .addGrid(randomForest.maxDepth, [5, 10, 15])\
    .addGrid(randomForest.maxBins, [32, 64, 128])\
    .build()

# Definiamo il CrossValidator
crossValidator = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse"),
    numFolds=3,
    parallelism=4,
    seed=48
)

# Utilizzando il metodo fit della pipeline, otterremo un Transformer, che utilizzeremo per predire i prezzi delle case.
cvModel = crossValidator.fit(train_data)

# Utilizzando il metodo transform del Transformer prevediamo i prezzi delle case della parte di dataset che abbiamo riservato al testing.
predictions = cvModel.transform(test_data)

# Dobbiamo quindi calcolare il logaritmo delle previsioni che abbiamo effettuato per poter valutare il modello utilizzando la radice dell'errore quadratico medio calcolato sul logaritmo del prezzo previsto e sul logaritmo del prezzo effettivo. 
predictions = predictions.withColumn("exp_prediction", f.expm1(predictions["prediction"]))

# Utilizziamo il RegressionEvaluator per calcolare la radice dell'errore quadratico medio sui logaritmi delle previsioni. Questo ci consente di trattare in un modo uniforme sia gli errori sulle case più economiche che su quelle più costose rispetto alla nostra predizione.  
evaluator_log = RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse")
rmse = evaluator_log.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) between Logarithms of SalePrice: {rmse}")

evaluator_raw = RegressionEvaluator(labelCol="SalePrice", predictionCol="exp_prediction", metricName="rmse")
rmse_raw = evaluator_raw.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on original SalePrice: {rmse_raw}")


Root Mean Squared Error (RMSE) between Logarithms of SalePrice: 0.14202610220403267


Root Mean Squared Error (RMSE) on original SalePrice: 25911.914736698385


Il risultato ottenuto in questo caso è decisamente più soddisfacente, complice sicuramente l'aver ottimizzato alcuni iperparametri. Nello specifico gli iperparametri che abbiamo considerato sono i seguenti:
- numTrees, utilizzare un maggior numero di alberi nel nostro algoritmo RandomForest migliora l'accuratezza delle previsioni perchè riduce la varianza del modello, a scapito però di un maggior tempo di addestramento.
- maxDepth, ci permette di indicare la profondità massima per ogni albero del nostro algoritmo RandomForest; con una maggiore profondità possiamo migliorare l'accuratezza del modello, bisogna però anche considerare che un valore eccessivo in questo parametro potrebbe portarci a problemi di overfitting (specie su dataset rumorosi).
- maxBins, ovvero il numero di categorie da generare per ogni feature continua o categorica; aumentare questo valore ci permette di migliorare la capacità del nostro modello di trovare soglie ottimali per gli split nei diversi alberi del nostro ensemble di Decision Tree, a scapito però di un maggior costo computazionale e un maggior tempo di addestramento.

In base ai valori migliori di ogni iperparametro, impostiamo i successivi iperparametri da testare nel crossValidator. 

In [26]:
best_model = cvModel.bestModel  
# Recuperiamo il modello Random Forest dalla pipeline
rf_model = best_model.stages[-1]
# Quindi stampiamo a schermo i parametri migliori
print("Parametri migliori del Random Forest:")
print(f"numTrees: {rf_model.getNumTrees}")
print(f"maxDepth: {rf_model.getMaxDepth()}")
print(f"maxBins: {rf_model.getMaxBins()}")

Parametri migliori del Random Forest:
numTrees: 200
maxDepth: 10
maxBins: 32


Per cercare di migliorare ulteriormente i risultati del nostro modello, cerchiamo di migliorare la scelta del sottoinsieme delle features da considerare in fase di addestramento. Per fare questo andiamo a misurare la correlazione tra tutte le possibili categorie e la feature target SalePrice, misurando così il grado di relazione lineare tra le variabili e SalePrice; valori vicini ad 1 indicheranno una correlazione positiva, cioè che quando una variabile tenda ad aumentare l'altra tende a sua volta ad aumentare, viceversa valori vicini a -1 saranno altrettanto importanti perchè ci indicheranno che quando una variabile tende ad aumentare l'altra tende a diminuire (per questo motivo considereremo il valore assoluto della correlazione nel confrontarla con il valore soglia); valori di correlazione vicini a 0 indicheranno che non vi è alcuna relazione lineare tra le variabili.

In [27]:
# Calcoliamo la correlazione per ogni feature quantitativa con SalePrice
correlations = []
for feature in quantitative:
    correlation = trainDF.select(corr(feature, "SalePrice").alias("correlation")).collect()[0]["correlation"]
    correlations.append((feature, correlation))

correlation_df = spark.createDataFrame(correlations, ["Feature", "Correlation"])
most_correlated=correlation_df.filter(abs(correlation_df["Correlation"])>0.5).collect()
most_correlated=[feature for feature, correlation in most_correlated]
print(most_correlated)

['OverallQual', 'YearBuilt', 'YearRemodAdd', 'TotalBsmtSF', '1stFlrSF', 'GrLivArea', 'FullBath', 'TotRmsAbvGrd', 'GarageCars', 'GarageArea']


Combiniamo quindi la lista delle features con un maggiore livello di importanza secondo l'algoritmo RandomForest con le features che hanno una maggiore correlazione con SalePrice, e proviamo ad addestrare nuovamente il nostro modello.

In [28]:
# Combiniamo le feature selezionate
combined_features = list(set(most_correlated + important_feature))

print(f"Feature combinate: {combined_features}")

Feature combinate: ['1stFlrSF', 'ExterQual_index', 'TotalBsmtSF', 'BsmtQual_index', 'TotRmsAbvGrd', 'CentralAir_index', 'FullBath', 'BsmtFinSF1', 'KitchenQual_index', 'Neighborhood_index', 'GarageCars', 'GrLivArea', 'GarageArea', 'YearBuilt', 'YearRemodAdd', 'OverallQual']


Notiamo che nella lista delle features combinate sono presenti sia la feature "GarageArea" che la feature "GarageCars"; entrambe ci forniscono un'informazione simile circa la dimensione del garage, per cui scegliamo di tenere soltanto "GarageCars".
Potremmo fare lo stesso ragionamento anche con "YearBuilt" e "YearRemodAdd", dato che ci forniscono informazioni simili circa l'anno di costruzione o ristrutturazione della casa, e in alcuni casi indicano lo stesso valore quando non vi è stata alcuna ristrutturazione; ma avendo fatto diverse prove i risultati migliori sono stati ottenuti mantenendo entrambe le categorie.

In [29]:
#combined_features.remove("YearBuilt")
combined_features.remove("GarageArea")

In [30]:
assembler = VectorAssembler(inputCols=combined_features, outputCol="features")

# Aggiungiamo la colonna logaritmica log_SalePrice che ci servirà per calcolare l'errore
trainDF=trainDF.withColumn("log_SalePrice", f.log1p(trainDF["SalePrice"]))

# Dividiamo il dataset in una parte destinata al training ed una parte per il test
train_data, test_data = trainDF.randomSplit([0.8, 0.2], seed=48)

# Utilizziamo il regressore Random Forest come Estimator, impostando il vettore delle feature e la label che vogliamo predire.
randomForest = RandomForestRegressor(featuresCol="features", labelCol="log_SalePrice", seed=48)

# Impostiamo la pipeline di trasformazione specificando gli stages di lavoro che dovremo seguire.
# assembliamo le features in un unico vettore con l'assembler, ed utilizziamo l'estimator randomForest per addestrare un modello utilizzando il vettore delle features per prevedere SalePrice. 
pipeline = Pipeline(stages=[assembler, randomForest])

# Grid dei parametri per il tuning
paramGrid = ParamGridBuilder()\
    .addGrid(randomForest.numTrees, [50, 100, 150])\
    .addGrid(randomForest.maxDepth, [10, 15, 20])\
    .addGrid(randomForest.maxBins, [32, 64, 128])\
    .build()

# Definiamo il CrossValidator
crossValidator = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse"),
    numFolds=3,  # Numero di fold per il cross-validation
    parallelism=4,  # per sfruttare più core durante il processo
    seed=48
)

# Utilizzando il metodo fit della pipeline, otterremo un Transformer, che utilizzeremo per predire i prezzi delle case.
cvModel = crossValidator.fit(train_data)

# Utilizzando il metodo transform del Transformer prevediamo i prezzi delle case della parte di dataset che abbiamo riservato al testing.
predictions = cvModel.transform(test_data)

# Dobbiamo quindi calcolare il logaritmo delle previsioni che abbiamo effettuato per poter valutare il modello utilizzando la radice dell'errore quadratico medio calcolato sul logaritmo del prezzo previsto e sul logaritmo del prezzo effettivo. 
#predictions = predictions.withColumn("log_prediction", f.log1p(predictions["prediction"]))
predictions = predictions.withColumn("exp_prediction", f.expm1(predictions["prediction"]))

# Utilizziamo il RegressionEvaluator per calcolare la radice dell'errore quadratico medio sui logaritmi delle previsioni. Questo ci consente di trattare in un modo uniforme sia gli errori sulle case più economiche che su quelle più costose rispetto alla nostra predizione.  
evaluator = RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) between Logarithms of SalePrice: {rmse}")

evaluator_raw = RegressionEvaluator(labelCol="SalePrice", predictionCol="exp_prediction", metricName="rmse")
rmse_raw = evaluator_raw.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on original SalePrice: {rmse_raw}")


Root Mean Squared Error (RMSE) between Logarithms of SalePrice: 0.13860956995624835


Root Mean Squared Error (RMSE) on original SalePrice: 24992.898240058927


Facciamo un'ulteriore considerazione circa le colonne "TotalBsmtSF" e "1stFlrSF", entrambe contenute nella nostra lista delle features più rilevanti. Entrambe ci danno informazioni sulla superficie (in piedi quadrati, come tutte le misure del dataset) rispettivamente del seminterrato e del piano terra; si è pensato allora di calcolare una feature "Total_square" che indichi la superficie totale dell'abitazione, ottenuta sommando la superficie del seminterrato (TotalBsmtSF), del piano terra (1stFlrSF) e del primo piano (2ndFlrSF). 

In [31]:
trainDF = trainDF.withColumn("Total_square", f.col("TotalBsmtSF") + f.col("1stFlrSF") + f.col("2ndFlrSF"))
combined_features.remove("TotalBsmtSF")
combined_features.remove("1stFlrSF")
combined_features.append("Total_square")

print(combined_features)


['ExterQual_index', 'BsmtQual_index', 'TotRmsAbvGrd', 'CentralAir_index', 'FullBath', 'BsmtFinSF1', 'KitchenQual_index', 'Neighborhood_index', 'GarageCars', 'GrLivArea', 'YearBuilt', 'YearRemodAdd', 'OverallQual', 'Total_square']


In [32]:
assembler = VectorAssembler(inputCols=combined_features, outputCol="features")

# Aggiungiamo la colonna logaritmica log_SalePrice che ci servirà per calcolare l'errore
trainDF=trainDF.withColumn("log_SalePrice", f.log1p(trainDF["SalePrice"]))

# Dividiamo il dataset in una parte destinata al training ed una parte per il test
train_data, test_data = trainDF.randomSplit([0.8, 0.2], seed=48)

# Utilizziamo il regressore Random Forest come Estimator, impostando il vettore delle feature e la label che vogliamo predire.
randomForest = RandomForestRegressor(featuresCol="features", labelCol="log_SalePrice", seed=48)

# Impostiamo la pipeline di trasformazione specificando gli stages di lavoro che dovremo seguire.
# quindi assembliamo le features in un unico vettore con l'assembler, ed utilizziamo l'estimator randomForest per addestrare un modello utilizzando il vettore delle features per prevedere SalePrice. 
pipeline = Pipeline(stages=[assembler, randomForest])

# Grid dei parametri per il tuning
paramGrid = ParamGridBuilder()\
    .addGrid(randomForest.numTrees, [50, 100, 150])\
    .addGrid(randomForest.maxDepth, [10, 15, 20])\
    .addGrid(randomForest.maxBins, [32, 64, 128])\
    .build()

# Definiamo il CrossValidator
crossValidator = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse"),
    numFolds=3,  # Numero di fold per il cross-validation
    parallelism=4,  # per sfruttare più core durante il processo
    seed=48
)

# Utilizzando il metodo fit della pipeline, otterremo un Transformer, che utilizzeremo per predire i prezzi delle case.
cvModel = crossValidator.fit(train_data)

# Utilizzando il metodo transform del Transformer prevediamo i prezzi delle case della parte di dataset che abbiamo riservato al testing.
predictions = cvModel.transform(test_data)

# Dobbiamo quindi calcolare il logaritmo delle previsioni che abbiamo effettuato per poter valutare il modello utilizzando la radice dell'errore quadratico medio calcolato sul logaritmo del prezzo previsto e sul logaritmo del prezzo effettivo. 
#predictions = predictions.withColumn("log_prediction", f.log1p(predictions["prediction"]))
predictions = predictions.withColumn("exp_prediction", f.expm1(predictions["prediction"]))

# Utilizziamo il RegressionEvaluator per calcolare la radice dell'errore quadratico medio sui logaritmi delle previsioni. Questo ci consente di trattare in un modo uniforme sia gli errori sulle case più economiche che su quelle più costose rispetto alla nostra predizione.  
evaluator = RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) between Logarithms of SalePrice: {rmse}")

evaluator_raw = RegressionEvaluator(labelCol="SalePrice", predictionCol="exp_prediction", metricName="rmse")
rmse_raw = evaluator_raw.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on original SalePrice: {rmse_raw}")


Root Mean Squared Error (RMSE) between Logarithms of SalePrice: 0.13798082868079434


Root Mean Squared Error (RMSE) on original SalePrice: 24571.902508332463


In [33]:
trainDF.describe()["summary", "SalePrice"].show()

+-------+------------------+
|summary|         SalePrice|
+-------+------------------+
|  count|              1460|
|   mean|180921.19589041095|
| stddev| 79442.50288288663|
|    min|             34900|
|    max|            755000|
+-------+------------------+


Possiamo ritenerci relativamente soddisfatti di quest'ultimo risultato ottenuto sul set di validazione (il 20% del training set tenuto da parte), abbiamo infatti un errore nell'ordine di 25000 rispetto al prezzo previsto. Confrontando questo risultato con i valori medi della variabile SalePrice nel dataset notiamo che il nostro RMSE rappresenta circa il 13,6% rispetto al costo medio di un'abitazione. Il nostro algoritmo potrebbe quindi essere sufficientemente preciso per stimare in modo aggregato il prezzo medio di un dataset di abitazioni, resta invece poco preciso nel caso in cui volessimo stimare con chiarezza il prezzo di singole abitazioni (infatti specie se molto economiche notiamo come l'errore sarebbe in questo caso troppo elevato).
Quello che ci interessa maggiormente però è considerare RMSE tra il logaritmo del prezzo previsto e il logaritmo del prezzo reale; infatti considerare l'errore nel dominio logaritmico è utile quando ci interessa ottenere un'indicazione che sia indipendente dalla scala dei dati (ed in questo caso abbiamo a che fare sia con valori di abitazioni poco costose che con valori di abitazioni molto costose). Effettuando un'esponenziazione del valore dell'RMSE logaritmico, otteniamo un fattore moltiplicativo medio dell'errore pari a 1.1480, che corrisponde ad un errore percentuale sul prezzo reale di circa il 14.8%. 
Proviamo ad analizzare i risultati ottenuti quando utilizziamo il modello con dataset diversi da quello di addestramento. 

In [34]:
testDF = testDF.withColumn("Total_square", f.col("TotalBsmtSF") + f.col("1stFlrSF") + f.col("2ndFlrSF"))

predictions = cvModel.transform(testDF)

# Il risultato è piccolo (1459 righe): lo raccogliamo sul driver e lo salviamo come singolo CSV da caricare su Kaggle
os.makedirs(OUTPUT_DIR, exist_ok=True)
predictions \
  .withColumn("SalePrice", f.expm1(predictions["prediction"])) \
  .select("Id", "SalePrice") \
  .orderBy("Id") \
  .toPandas() \
  .to_csv(f"{OUTPUT_DIR}/saleprice_prediction_randomforest.csv", index=False)


Il valore che Kaggle ci restituisce quando proviamo a fare la submission è di 0.14719. Rispetto al valore che abbiamo ottenuto sul set di validazione, in questo caso l'errore è più alto; tuttavia non essendo eccessivo il divario tra i due valori (c'è uno scostamento del 6.7% circa se consideriamo i due valori di errore: (0.14719 - 0.13798) / 0.13798 * 100) non c'è un overfitting rilevante e possiamo affermare che il modello riesce a generalizzare in modo accettabile anche con dati diversi da quelli di training.

Proviamo adesso ad utilizzare un algoritmo di addestramento alternativo al RandomForest, ovvero XGBoost. XGBoost implementa un algoritmo di gradient boosting, una tecnica che lavora con un ensemble di alberi decisionali in modo sequenziale (facendo in modo che ogni nuovo gruppo cerchi di correggere gli errori residui dei modelli precedenti) cercando di ottimizzare una loss function utilizzando il metodo del gradiente. 
Rispetto al Random Forest l'algoritmo dovrebbe presentare la possibilità di gestire in maniera più precisa gli iperparametri, facendo corrispondere alla necessità di un tuning più complicato prestazioni generalmente migliori e tempi di esecuzione più brevi.
Gli iperparametri che andremo a considerare sono:
- max_depth, controlla la profondità massima degli alberi. Aumentare questo valore permette agli alberi di catturare maggiormente i legami statistici tra le features, però un valore eccessivo puo' portare a problemi di overfitting.  
- learning_rate, imposta la velocità con cui il modello apprende (impostare dei valori più bassi rende sì l'apprendimento più lento, ma dovrebbe essere anche più preciso)
- n_estimators, indica il numero di alberi che utilizziamo nel modello; con un numero maggiore di alberi dovremmo migliorare le performance fino a quando il modello inizia a convergere, ma ogni albero aggiuntivo richiede risorse e tempi di calcolo aggiuntivi ed inoltre con un numero eccessivo il modello puo' sovradattarsi ai dati. 

Successivamente proveremo a fare il tuning considerando anche:
- subsample, controlla la percentuale di campioni del dataset che vengono utilizzati per ogni albero; ridurre questo valore ci aiuta a prevenire fenomeni di overfitting, tuttavia valori insufficienti potrebbero causare problemi di underfitting.
- colsample_bytree, controlla la frazione di features che vengono utilizzate per ogni albero; aumentare questo valore fa in modo che il modello consideri un maggior numero di features per ogni albero (con 1 le consideriamo tutte); anche in questo caso bisogna cercare di scegliere il valore migliore per evitare l'overfitting.
    

Proviamo quindi ad addestrare un modello XGBoost utilizzando le features che abbiamo selezionato per il RandomForest. Sebbene alcune di queste feature siano state scelte utilizzando l'importanza che il RandomForest attribuiva ad ogni caratteristica, dovremmo comunque ottenere dei buoni risultati dato che comunque l'algoritmo dovrebbe aver selezionato le features più rilevanti per effettuare la predizione del prezzo delle abitazioni. 
Confrontoremo comunque i risultati ottenuti eseguendo nuovamente la pre-selezione delle features anche con XGBoost.

In [35]:
print(combined_features)

['ExterQual_index', 'BsmtQual_index', 'TotRmsAbvGrd', 'CentralAir_index', 'FullBath', 'BsmtFinSF1', 'KitchenQual_index', 'Neighborhood_index', 'GarageCars', 'GrLivArea', 'YearBuilt', 'YearRemodAdd', 'OverallQual', 'Total_square']


In [36]:
os.environ['XGBOOST_VERBOSITY'] = '0'


In [37]:
# Assembliamo le features in un'unica colonna
assembler = VectorAssembler(inputCols=combined_features, outputCol="features")

# Configuriamo XGBoost
xgb = SparkXGBRegressor(
    features_col="features",
    label_col="log_SalePrice",
    objective="reg:squarederror",
    num_workers=4,
    verbose=False,
    seed=48,
)

# Grid dei parametri per il tuning. Inizialmente consideriamo solo max_depth, learning_rate e n_estimators
paramGrid = ParamGridBuilder()\
    .addGrid(xgb.max_depth, [3, 5, 10])\
    .addGrid(xgb.learning_rate, [0.1, 0.05, 0.025])\
    .addGrid(xgb.n_estimators, [25, 50, 100])\
    .build()

# Definiamo il CrossValidator
evaluator = RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse")
crossValidator = CrossValidator(
    estimator=xgb,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=4,
    seed=48
)

# Impostiamo la pipeline di trasformazione specificando gli stages di lavoro che dovremo seguire.
# Assembliamo le features in un unico vettore con l'assembler, ed utilizziamo l'estimator XGBooster per addestrare un modello utilizzando il vettore delle features per prevedere SalePrice. 
pipeline = Pipeline(stages=[assembler, crossValidator])

# Dividiamo il dataset in una parte destinata al training ed una parte per il test
train_data, test_data = trainDF.randomSplit([0.8, 0.2], seed=48)

# Utilizzando il metodo fit della pipeline, otterremo un Transformer, che utilizzeremo per predire i prezzi delle case.
cvModel = pipeline.fit(train_data)

# Utilizzando il metodo transform del Transformer prevediamo i prezzi delle case della parte di dataset che abbiamo riservato al testing.
predictions = cvModel.transform(test_data)

# Utilizziamo il RegressionEvaluator per calcolare la radice dell'errore quadratico medio sui logaritmi delle previsioni. Questo ci consente di trattare in un modo uniforme sia gli errori sulle case più economiche che su quelle più costose rispetto alla nostra predizione.  
evaluator = RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on log(SalePrice): {rmse}")

# Calcoliamo anche la radice dell'errore quadratico medio sulle previsioni rispetto ai prezzi reali.
predictions = predictions.withColumn("exp_prediction", f.expm1(predictions["prediction"]))
evaluator_raw = RegressionEvaluator(labelCol="SalePrice", predictionCol="exp_prediction", metricName="rmse")
rmse_raw = evaluator_raw.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on original SalePrice: {rmse_raw}")


Root Mean Squared Error (RMSE) on log(SalePrice): 0.13195025497168075


Root Mean Squared Error (RMSE) on original SalePrice: 27805.01828269359


In [38]:
predictions = cvModel.transform(testDF)

# Il risultato è piccolo (1459 righe): lo raccogliamo sul driver e lo salviamo come singolo CSV da caricare su Kaggle
os.makedirs(OUTPUT_DIR, exist_ok=True)
predictions \
  .withColumn("SalePrice", f.expm1(predictions["prediction"])) \
  .select("Id", "SalePrice") \
  .orderBy("Id") \
  .toPandas() \
  .to_csv(f"{OUTPUT_DIR}/saleprice_prediction_xgb2.csv", index=False)


Il risultato ottenuto a prima vista sembrerebbe essere un miglioramento rispetto ai modelli precedenti, ed in effetti abbiamo ottenuto un RMSE sul logaritmo del prezzo delle abitazioni di 0.13195, migliore dei precedenti. Se proviamo ad utilizzare il modello su un dataset differente da quello di training, come testDF, Kaggle ci restituisce un punteggio di 0.14229. Il risultato non indica un overfitting rilevante (c'è uno scostamento del 7.8% circa se consideriamo (0.14229 - 0.13195) / 0.13195 * 100; più elevato di prima ma non ancora indice di un overfitting eccessivo); il nostro modello generalizza abbastanza bene, ed il risultato è leggermente migliore di quello ottenuto con il RandomForest (0.14229 contro 0.14719).
Proviamo a fare in modo che sia l'algoritmo XGBoost stesso a calcolare direttamente i valori di importanza delle features e proviamo a migliorare le performance del nostro modello.

In [39]:
# Assembliamo le features in un'unica colonna
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

# Configuriamo XGBoost
xgb = SparkXGBRegressor(
    features_col="features",
    label_col="log_SalePrice",
    objective="reg:squarederror",
    num_workers=4,
    verbosity=0,
    verbose=False,
    seed=48,
)

# Impostiamo la pipeline di trasformazione specificando gli stages di lavoro che dovremo seguire.
# Assembliamo le features in un unico vettore con l'assembler, ed utilizziamo l'estimator XGBooster per addestrare un modello utilizzando il vettore delle features per prevedere SalePrice. 
pipeline = Pipeline(stages=[assembler, xgb])

# Grid dei parametri per il tuning. Inizialmente consideriamo solo max_depth, learning_rate e n_estimators
paramGrid = ParamGridBuilder()\
    .addGrid(xgb.max_depth, [5, 10, 15])\
    .addGrid(xgb.learning_rate, [0.1, 0.05, 0.025])\
    .addGrid(xgb.n_estimators, [50, 100, 150])\
    .build()

# Definiamo il CrossValidator
evaluator = RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse")
crossValidator = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=4,
    seed=48
)

# Dividiamo il dataset in una parte destinata al training ed una parte per il test
train_data, test_data = trainDF.randomSplit([0.8, 0.2], seed=48)

# Utilizzando il metodo fit, otterremo un Transformer, che utilizzeremo per predire i prezzi delle case.
cvModel = crossValidator.fit(train_data)

# Utilizzando il metodo transform del Transformer prevediamo i prezzi delle case della parte di dataset che abbiamo riservato al testing.
predictions = cvModel.transform(test_data)

# Utilizziamo il RegressionEvaluator per calcolare la radice dell'errore quadratico medio sui logaritmi delle previsioni. Questo ci consente di trattare in un modo uniforme sia gli errori sulle case più economiche che su quelle più costose rispetto alla nostra predizione.  
evaluator = RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on log(SalePrice): {rmse}")

# Calcoliamo anche la radice dell'errore quadratico medio sulle previsioni rispetto ai prezzi reali.
predictions = predictions.withColumn("exp_prediction", f.expm1(predictions["prediction"]))
evaluator_raw = RegressionEvaluator(labelCol="SalePrice", predictionCol="exp_prediction", metricName="rmse")
rmse_raw = evaluator_raw.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on original SalePrice: {rmse_raw}")

# Analizziamo i migliori iperparametri trovati dal CrossValidator per effettuare il tuning. 
best_model = cvModel.bestModel
xgb_model = best_model.stages[-1]
print("Best parameters:")
print(f"Max Depth: {xgb_model.getOrDefault('max_depth')}")
print(f"Learning Rate: {xgb_model.getOrDefault('learning_rate')}")
print(f"Number of Estimators: {xgb_model.getOrDefault('n_estimators')}")


Root Mean Squared Error (RMSE) on log(SalePrice): 0.13703390576265567


Root Mean Squared Error (RMSE) on original SalePrice: 36392.43383885227
Best parameters:
Max Depth: 5
Learning Rate: 0.1
Number of Estimators: 150


In [40]:
# estraiamo il modello XGBoost dal CrossValidator
xgb_model = cvModel.bestModel.stages[-1]
booster = xgb_model.get_booster()

# otteniamo l'importanza delle feature e mappiamo il risultato con i nomi delle colonne corretti
feature_importance = booster.get_score(importance_type='weight')
# get_score restituisce chiavi del tipo "f0", "f1", ... (indice della feature nel vettore) e
# omette le feature mai usate negli split: ricaviamo il nome dalla posizione indicata nella chiave
feature_names = assembler.getInputCols()
feature_importance_mapped = {feature_names[int(key[1:])]: value for key, value in feature_importance.items()}
feature_importance_mapped = dict(sorted(feature_importance_mapped.items(), key=lambda x: x[1], reverse=True))

# quindi filtriamo le features sopra una certa soglia
filtered_features = [feature for feature, importance in feature_importance_mapped.items() if importance > 80]
for feature, importance in feature_importance_mapped.items():
    print(f"{feature}: {importance}")
print(f"Feature selezionate: {filtered_features}")

# e uniamo la lista delle features più importanti a quella delle features maggiormente correlate con SalePrice
xgb_features=list(set(most_correlated + filtered_features))

LotFrontage: 276.0
LotArea: 190.0
BsmtFinSF1: 169.0
YearBuilt: 165.0
GrLivArea: 162.0
GarageArea: 161.0
MSSubClass: 158.0
TotalBsmtSF: 140.0
OverallQual: 126.0
YearRemodAdd: 124.0
1stFlrSF: 121.0
Neighborhood_index: 115.0
BsmtUnfSF: 114.0
OverallCond: 113.0
OpenPorchSF: 87.0
MoSold: 83.0
Exterior1st_index: 81.0
WoodDeckSF: 77.0
2ndFlrSF: 76.0
SaleCondition_index: 63.0
MasVnrArea: 53.0
YrSold: 51.0
MSZoning_index: 50.0
EnclosedPorch: 39.0
Condition1_index: 39.0
BsmtFinSF2: 38.0
BsmtFinType1_index: 36.0
GarageType_index: 33.0
Functional_index: 32.0
BsmtFullBath: 31.0
GarageCars: 31.0
LotConfig_index: 31.0
ScreenPorch: 30.0
PavedDrive_index: 30.0
ExterCond_index: 28.0
BsmtExposure_index: 27.0
RoofStyle_index: 26.0
Exterior2nd_index: 26.0
TotRmsAbvGrd: 25.0
Fireplaces: 25.0
BedroomAbvGr: 24.0
SaleType_index: 23.0
KitchenQual_index: 19.0
CentralAir_index: 18.0
Electrical_index: 18.0
GarageFinish_index: 18.0
LotShape_index: 16.0
FireplaceQu_index: 16.0
FullBath: 15.0
LandContour_index: 15.0


Proviamo quindi ad addestrare il nostro modello considerando unicamente le features selezionate.

In [41]:
# Assembliamo le features in un'unica colonna
assembler = VectorAssembler(inputCols=xgb_features, outputCol="features")

# Configuriamo XGBoost
xgb = SparkXGBRegressor(
    features_col="features",
    label_col="log_SalePrice",
    objective="reg:squarederror",
    num_workers=4,
    verbosity=0,
    verbose=False,
    seed=48,
)

# Impostiamo la pipeline di trasformazione specificando gli stages di lavoro che dovremo seguire.
# Assembliamo le features in un unico vettore con l'assembler, ed utilizziamo l'estimator XGBooster per addestrare un modello utilizzando il vettore delle features per prevedere SalePrice. 
pipeline = Pipeline(stages=[assembler, xgb])

# Grid dei parametri per il tuning. Consideriamo ora anche altri iperparametri.
paramGrid = ParamGridBuilder()\
    .addGrid(xgb.max_depth, [3, 5, 7])\
    .addGrid(xgb.learning_rate, [0.15, 0.1, 0.05])\
    .addGrid(xgb.n_estimators, [50, 100, 150])\
    .build()
    #.addGrid(xgb.subsample, [0.8, 0.5])\
    #.addGrid(xgb.colsample_bytree, [0.6, 0.8])\
    #.build()

# Definiamo il CrossValidator
evaluator = RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse")
crossValidator = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=4,
    seed=48
)

# Dividiamo il dataset in una parte destinata al training ed una parte per il test
train_data, test_data = trainDF.randomSplit([0.8, 0.2], seed=48)

# Utilizzando il metodo fit otterremo un Transformer che utilizzeremo per predire i prezzi delle case.
cvModel = crossValidator.fit(train_data)

# Utilizzando il metodo transform del Transformer prevediamo i prezzi delle case della parte di dataset che abbiamo riservato al testing.
predictions = cvModel.transform(test_data)

# Utilizziamo il RegressionEvaluator per calcolare la radice dell'errore quadratico medio sui logaritmi delle previsioni. Questo ci consente di trattare in un modo uniforme sia gli errori sulle case più economiche che su quelle più costose rispetto alla nostra predizione.  
evaluator = RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on log(SalePrice): {rmse}")

# Calcoliamo anche la radice dell'errore quadratico medio sulle previsioni rispetto ai prezzi reali.
predictions = predictions.withColumn("exp_prediction", f.expm1(predictions["prediction"]))
evaluator_raw = RegressionEvaluator(labelCol="SalePrice", predictionCol="exp_prediction", metricName="rmse")
rmse_raw = evaluator_raw.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on original SalePrice: {rmse_raw}")

# Analizziamo i migliori iperparametri trovati dal CrossValidator per effettuare il tuning. 
best_model = cvModel.bestModel
xgb_model = best_model.stages[-1]
print("Best parameters:")
print(f"Max Depth: {xgb_model.getOrDefault('max_depth')}")
print(f"Learning Rate: {xgb_model.getOrDefault('learning_rate')}")
print(f"Number of Estimators: {xgb_model.getOrDefault('n_estimators')}")
print(f"Subsample: {xgb_model.getOrDefault('subsample')}")
print(f"Colsample ByTree: {xgb_model.getOrDefault('colsample_bytree')}")


Root Mean Squared Error (RMSE) on log(SalePrice): 0.1277689497779637


Root Mean Squared Error (RMSE) on original SalePrice: 29435.592048208673
Best parameters:
Max Depth: 3
Learning Rate: 0.1
Number of Estimators: 150
Subsample: None
Colsample ByTree: None


Riduciamo ulteriormente le features selezionate, utilizzando nuovamente il valore di importanza assegnato dall'ultimo modello che abbiamo addestrato.

In [42]:
# estraiamo il modello XGBoost dal CrossValidator
xgb_model = cvModel.bestModel.stages[-1]
booster = xgb_model.get_booster()

# otteniamo l'importanza delle feature e mappiamo il risultato con i nomi delle colonne corretti
feature_importance = booster.get_score(importance_type='weight')
# get_score restituisce chiavi del tipo "f0", "f1", ... (indice della feature nel vettore) e
# omette le feature mai usate negli split: ricaviamo il nome dalla posizione indicata nella chiave
feature_names = assembler.getInputCols()
feature_importance_mapped = {feature_names[int(key[1:])]: value for key, value in feature_importance.items()}
feature_importance_mapped = dict(sorted(feature_importance_mapped.items(), key=lambda x: x[1], reverse=True))

# quindi filtriamo le features sopra una certa soglia
xgb_features2 = [feature for feature, importance in feature_importance_mapped.items() if importance > 20]
for feature, importance in feature_importance_mapped.items():
    print(f"{feature}: {importance}")
print(f"Feature selezionate: {xgb_features2}")

YearBuilt: 93.0
GrLivArea: 85.0
BsmtFinSF1: 84.0
LotArea: 71.0
TotalBsmtSF: 67.0
1stFlrSF: 63.0
GarageArea: 61.0
OverallCond: 60.0
BsmtUnfSF: 58.0
OpenPorchSF: 57.0
Neighborhood_index: 57.0
OverallQual: 44.0
LotFrontage: 40.0
YearRemodAdd: 35.0
Exterior1st_index: 34.0
MoSold: 33.0
MSSubClass: 26.0
GarageCars: 19.0
FullBath: 13.0
TotRmsAbvGrd: 11.0
Feature selezionate: ['YearBuilt', 'GrLivArea', 'BsmtFinSF1', 'LotArea', 'TotalBsmtSF', '1stFlrSF', 'GarageArea', 'OverallCond', 'BsmtUnfSF', 'OpenPorchSF', 'Neighborhood_index', 'OverallQual', 'LotFrontage', 'YearRemodAdd', 'Exterior1st_index', 'MoSold', 'MSSubClass']


Effettuiamo quindi gli stessi ragionamenti che abbiamo fatto nel caso dell'algoritmo RandomForest anche per XGBoost, aggiungendo la variabile YrSold nonostante sia presente anche YearBuilt, e aggiungiamo la variabile Total_square al posto delle variabili 1stFlrSF e TotalBsmtSF. 
Quindi addestriamo nuovamente il nostro modello.

In [43]:
if "YrSold" not in xgb_features2:
    xgb_features2.append("YrSold")
for feature in ["1stFlrSF", "TotalBsmtSF", "2ndFlrSF"]:
    if feature in xgb_features2:
        xgb_features2.remove(feature)
xgb_features2.append("Total_square")


In [44]:
print(xgb_features2)

['YearBuilt', 'GrLivArea', 'BsmtFinSF1', 'LotArea', 'GarageArea', 'OverallCond', 'BsmtUnfSF', 'OpenPorchSF', 'Neighborhood_index', 'OverallQual', 'LotFrontage', 'YearRemodAdd', 'Exterior1st_index', 'MoSold', 'MSSubClass', 'YrSold', 'Total_square']


In [45]:
# Assembliamo le features in un'unica colonna
assembler = VectorAssembler(inputCols=xgb_features2, outputCol="features")

# Configuriamo XGBoost
xgb = SparkXGBRegressor(
    features_col="features",
    label_col="log_SalePrice",
    objective="reg:squarederror",
    num_workers=4,
    verbosity=0,
    verbose=False,
    seed=48,
)

# Impostiamo la pipeline di trasformazione specificando gli stages di lavoro che dovremo seguire.
# Assembliamo le features in un unico vettore con l'assembler, ed utilizziamo l'estimator XGBooster per addestrare un modello utilizzando il vettore delle features per prevedere SalePrice. 
pipeline = Pipeline(stages=[assembler, xgb])

# Grid dei parametri per il tuning. Consideriamo ora anche altri iperparametri.
paramGrid = ParamGridBuilder()\
    .addGrid(xgb.max_depth, [3, 5, 7])\
    .addGrid(xgb.learning_rate, [0.15, 0.1, 0.05])\
    .addGrid(xgb.n_estimators, [100, 200, 250])\
    .addGrid(xgb.subsample, [0.8, 0.5])\
    .addGrid(xgb.colsample_bytree, [0.6, 0.8])\
    .build()

# Definiamo il CrossValidator
evaluator = RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse")
crossValidator = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=4,
    seed=48
)

# Dividiamo il dataset in una parte destinata al training ed una parte per il test
train_data, test_data = trainDF.randomSplit([0.8, 0.2], seed=48)

# Utilizzando il metodo fit otterremo un Transformer che utilizzeremo per predire i prezzi delle case.
cvModel = crossValidator.fit(train_data)

# Utilizzando il metodo transform del Transformer prevediamo i prezzi delle case della parte di dataset che abbiamo riservato al testing.
predictions = cvModel.transform(test_data)

# Utilizziamo il RegressionEvaluator per calcolare la radice dell'errore quadratico medio sui logaritmi delle previsioni. Questo ci consente di trattare in un modo uniforme sia gli errori sulle case più economiche che su quelle più costose rispetto alla nostra predizione.  
evaluator = RegressionEvaluator(labelCol="log_SalePrice", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on log(SalePrice): {rmse}")

# Calcoliamo anche la radice dell'errore quadratico medio sulle previsioni rispetto ai prezzi reali.
predictions = predictions.withColumn("exp_prediction", f.expm1(predictions["prediction"]))
evaluator_raw = RegressionEvaluator(labelCol="SalePrice", predictionCol="exp_prediction", metricName="rmse")
rmse_raw = evaluator_raw.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on original SalePrice: {rmse_raw}")

# Analizziamo i migliori iperparametri trovati dal CrossValidator per effettuare il tuning. 
best_model = cvModel.bestModel
xgb_model = best_model.stages[-1]
print("Best parameters:")
print(f"Max Depth: {xgb_model.getOrDefault('max_depth')}")
print(f"Learning Rate: {xgb_model.getOrDefault('learning_rate')}")
print(f"Number of Estimators: {xgb_model.getOrDefault('n_estimators')}")
print(f"Subsample: {xgb_model.getOrDefault('subsample')}")
print(f"Colsample ByTree: {xgb_model.getOrDefault('colsample_bytree')}")


Root Mean Squared Error (RMSE) on log(SalePrice): 0.12491928656761919


Root Mean Squared Error (RMSE) on original SalePrice: 27660.13073861966
Best parameters:
Max Depth: 5
Learning Rate: 0.05
Number of Estimators: 200
Subsample: 0.5
Colsample ByTree: 0.8


In [46]:
predictions = cvModel.transform(testDF)

# Il risultato è piccolo (1459 righe): lo raccogliamo sul driver e lo salviamo come singolo CSV da caricare su Kaggle
os.makedirs(OUTPUT_DIR, exist_ok=True)
predictions \
  .withColumn("SalePrice", f.expm1(predictions["prediction"])) \
  .select("Id", "SalePrice") \
  .orderBy("Id") \
  .toPandas() \
  .to_csv(f"{OUTPUT_DIR}/saleprice_prediction_xgb3.csv", index=False)


Il risultato ottenuto è il migliore tra quelli che siamo riusciti ad ottenere sul set di validazione; l'RMSE sul logaritmo del prezzo delle abitazioni è di 0.12492. 
Analizziamo ora la capacità di generalizzazione del modello con il dataset di test. Il valore che Kaggle ci restituisce è 0.13642, il migliore tra le submission effettuate. L'errore con il dataset di test non si discosta in maniera significativa rispetto al valore che abbiamo ottenuto sul set di validazione (c'è uno scostamento del 9.2% circa se consideriamo i due valori di errore: (0.13642 - 0.12492) / 0.12492 * 100). Possiamo quindi affermare che non c'è un overfitting rilevante e che il modello riesce a generalizzare bene anche con dati diversi da quelli di addestramento.